In [11]:
import os
import urllib.request
import scipy.io as sio
import numpy as np
import matplotlib.pyplot as plt
import requests
import time

In [12]:
# ==========================================
# 1. THE ROBUST DOWNLOADER FUNCTION
# ==========================================
def robust_download(url, save_path, max_retries=5):
    """Downloads a file in chunks and retries if the server drops the connection."""
    
    # Check if file already exists to save time on reruns
    if os.path.exists(save_path):
        print(f"[SKIP] File already exists: {save_path}")
        return

    for attempt in range(max_retries):
        try:
            print(f"Attempt {attempt + 1} of {max_retries}: Downloading from {url}...")
            
            # stream=True keeps the connection open and downloads in chunks
            response = requests.get(url, stream=True, timeout=30)
            response.raise_for_status() 
            
            total_size = int(response.headers.get('content-length', 0))
            downloaded_size = 0
            
            with open(save_path, 'wb') as file:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        file.write(chunk)
                        downloaded_size += len(chunk)
            
            # Verify we actually got the whole file to prevent the "bytes error"
            if total_size != 0 and downloaded_size < total_size:
                raise Exception(f"Incomplete download: {downloaded_size}/{total_size} bytes")
                
            print(f"[SUCCESS] Saved to {save_path}\n")
            return 
            
        except Exception as e:
            print(f"[FAILED] Error: {e}")
            if attempt < max_retries - 1:
                print("Server dropped connection. Retrying in 5 seconds...\n")
                time.sleep(5)
            else:
                print("Max retries reached. Please download manually via browser.\n")


# ==========================================
# 2. SETUP PATHS & VARIABLES
# ==========================================
# Use absolute pathing so the script runs from anywhere
main_path = os.path.join("..", "ds")

datasets = {
    'indian_pines': {
        'data_url': 'http://www.ehu.eus/ccwintco/uploads/6/67/Indian_pines_corrected.mat',
        'gt_url': 'http://www.ehu.eus/ccwintco/uploads/c/c4/Indian_pines_gt.mat',
        'data_file': 'indian_pines_corrected.mat',
        'gt_file': 'indian_pines_gt.mat'
    },
    'salinas_valley': {
        'data_url': 'http://www.ehu.eus/ccwintco/uploads/a/a3/Salinas_corrected.mat',
        'gt_url': 'http://www.ehu.eus/ccwintco/uploads/f/fa/Salinas_gt.mat',
        'data_file': 'salinas_valley_corrected.mat',
        'gt_file': 'salinas_valley_gt.mat'
    },
    'pavia_center': {
        'data_url': 'http://www.ehu.eus/ccwintco/uploads/e/e3/Pavia.mat',
        'gt_url': 'http://www.ehu.eus/ccwintco/uploads/5/53/Pavia_gt.mat',
        'data_file': 'pavia_center.mat',
        'gt_file': 'pavia_center_gt.mat'
    },
    'pavia_university': {
        'data_url': 'https://www.ehu.eus/ccwintco/uploads/e/ee/PaviaU.mat',
        'gt_url': 'https://www.ehu.eus/ccwintco/uploads/5/50/PaviaU_gt.mat',
        'data_file': 'pavia_university.mat',
        'gt_file': 'pavia_university_gt.mat'
    }
}

# ==========================================
# 3. CREATE DIRECTORIES & DOWNLOAD
# ==========================================
print("Starting dataset initialization...\n" + "="*50)

for folder_name, info in datasets.items():
    # 1. Create the specific dataset folder
    folder_path = os.path.join(main_path, folder_name)
    os.makedirs(folder_path, exist_ok=True)
    print(f"\n>>> Processing {folder_name.upper()} directory at '{folder_path}'")
    
    # 2. Define the exact save paths for the .mat files
    data_save_path = os.path.join(folder_path, info['data_file'])
    gt_save_path = os.path.join(folder_path, info['gt_file'])
    
    # 3. Execute the robust downloads
    robust_download(info['data_url'], data_save_path)
    robust_download(info['gt_url'], gt_save_path)

print("="*50 + "\nAll dataset directory structures and downloads completed!")

Starting dataset initialization...

>>> Processing INDIAN_PINES directory at '../ds/indian_pines'
Attempt 1 of 5: Downloading from http://www.ehu.eus/ccwintco/uploads/6/67/Indian_pines_corrected.mat...
[SUCCESS] Saved to ../ds/indian_pines/indian_pines_corrected.mat

Attempt 1 of 5: Downloading from http://www.ehu.eus/ccwintco/uploads/c/c4/Indian_pines_gt.mat...
[SUCCESS] Saved to ../ds/indian_pines/indian_pines_gt.mat


>>> Processing SALINAS_VALLEY directory at '../ds/salinas_valley'
Attempt 1 of 5: Downloading from http://www.ehu.eus/ccwintco/uploads/a/a3/Salinas_corrected.mat...
[SUCCESS] Saved to ../ds/salinas_valley/salinas_valley_corrected.mat

Attempt 1 of 5: Downloading from http://www.ehu.eus/ccwintco/uploads/f/fa/Salinas_gt.mat...
[SUCCESS] Saved to ../ds/salinas_valley/salinas_valley_gt.mat


>>> Processing PAVIA_CENTER directory at '../ds/pavia_center'
Attempt 1 of 5: Downloading from http://www.ehu.eus/ccwintco/uploads/e/e3/Pavia.mat...
[SUCCESS] Saved to ../ds/pavia_cent

In [2]:
# Download resources with GET request
def download_file(url, folder, filename):
    filepath = os.path.join(folder, filename)
    
    # Adding a User-Agent header because some university servers block blank requests
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    
    if not os.path.exists(filepath):
        print(f"Downloading {filename} into {folder}/...")
        with urllib.request.urlopen(req) as response, open(filepath, 'wb') as out_file:
            out_file.write(response.read())
        print("Done.")
    else:
        print(f"{filename} already exists in {folder}/.")
    return filepath

In [4]:
# 1 - SETUP THE PATH

# Indiana Pines
indian_pines = 'indian_pines'
url_indian_pines_data = 'http://www.ehu.eus/ccwintco/uploads/6/67/Indian_pines_corrected.mat'
url_indian_pines_gt = 'http://www.ehu.eus/ccwintco/uploads/c/c4/Indian_pines_gt.mat'
path_name_indian_pines_data = "indian_pines_corrected.mat"
path_name_indian_pines_gt = "indian_pines_gt.mat"

# Salinas Valley
salinas_valley = 'salinas_valley'
url_salinas_valley_data = 'http://www.ehu.eus/ccwintco/uploads/a/a3/Salinas_corrected.mat'
url_salinas_valley_gt = 'http://www.ehu.eus/ccwintco/uploads/f/fa/Salinas_gt.mat'
path_name_salinas_valley_data = "salinas_valley_corrected.mat"
path_name_salinas_valley_gt = "salinas_valley_gt.mat"

# Pavia Centre
pavia_center = 'pavia_center'
url_pavia_center_data = 'http://www.ehu.eus/ccwintco/uploads/e/e3/Pavia.mat'
url_pavia_center_gt = 'http://www.ehu.eus/ccwintco/uploads/5/53/Pavia_gt.mat'
path_name_pavia_center_data = "pavia_center.mat"
path_name_pavia_center_gt = "pavia_center_gt.mat"

# Pavia University
"""
Download the Data and the ground truth respectively from the following 2 links
https://www.ehu.eus/ccwintco/uploads/e/ee/PaviaU.mat
https://www.ehu.eus/ccwintco/uploads/5/50/PaviaU_gt.mat
"""


main_path = os.path.join("..", "ds")
folder_name_indian_pines = os.path.join(main_path, indian_pines)
folder_name_salinas_valley = os.path.join(main_path, salinas_valley)
folder_name_pavia_center = os.path.join(main_path, pavia_center)

if not os.path.exists(folder_name_indian_pines):
    os.makedirs(folder_name_indian_pines)
    print(f"Directory structure '{folder_name_indian_pines}' created.")
if not os.path.exists(folder_name_salinas_valley):
    os.makedirs(folder_name_salinas_valley)
    print(f"Directory structure '{folder_name_salinas_valley}' created.")
if not os.path.exists(folder_name_pavia_center):
    os.makedirs(folder_name_pavia_center)
    print(f"Directory structure '{folder_name_pavia_center}' created.")

In [5]:
# Indian Pines
download_file(url_indian_pines_data, folder_name_indian_pines, path_name_indian_pines_data)
download_file(url_indian_pines_gt, folder_name_indian_pines, path_name_indian_pines_gt)

IncompleteRead: IncompleteRead(4727933 bytes read, 1225594 more expected)

In [15]:
# Salinas Valley
download_file(url_salinas_valley_data, folder_name_salinas_valley, path_name_salinas_valley_data)
download_file(url_salinas_valley_gt, folder_name_salinas_valley, path_name_salinas_valley_gt)

salinas_valley_corrected.mat already exists in ../ds/salinas_valley/.
salinas_valley_gt.mat already exists in ../ds/salinas_valley/.


'../ds/salinas_valley/salinas_valley_gt.mat'

In [16]:
# Pavia Center
download_file(url_pavia_center_data, folder_name_pavia_center, path_name_pavia_center_data)
download_file(url_pavia_center_gt, folder_name_pavia_center, path_name_pavia_center_gt)

Done.
Done.


'../ds/pavia_centre/pavia_centre_gt.mat'